In [45]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
import requests

In [46]:
class QuakeEngine:
    def __init__(self):
        self.df = None
        self.scaler = StandardScaler()
        self.model = IsolationForest(contamination=0.05,
                                     random_state=42)
        self.pca = PCA(n_components=2,
                       random_state=42)
        self.base_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_month.geojson"

    def veri_cek_hazirla(self):
        response = requests.get(self.base_url)
        raw_data = response.json()

        deprem_listesi = []
        for deprem in raw_data['features']:
            mag = deprem['properties']['mag']
            sig = deprem['properties']['sig']
            boylam = deprem['geometry']['coordinates'][0]
            enlem = deprem['geometry']['coordinates'][1]
            derinlik = deprem['geometry']['coordinates'][2]
            yeni_deprem = {
                'mag' : mag,
                'sig' : sig,
                'boylam' : boylam,
                'enlem' : enlem,
                'derinlik' : derinlik
            }
            deprem_listesi.append(yeni_deprem)
                
        self.df = pd.DataFrame(deprem_listesi)

    def yapay_zeka_analizi(self):
        X = self.df[['mag','sig','derinlik']]
        X_scaled = self.scaler.fit_transform(X)
        X_pca = self.pca.fit_transform(X_scaled)
        self.model.fit(X_pca)
        predict = self.model.predict(X_pca)
        self.df['Anomali_Durumu'] = predict

    def eda_ozeti(self):
        return self.df.describe()
    
    def anomallikleri_getir(self):
        return self.df[self.df['Anomali_Durumu'] == -1]

In [47]:
class TerminalApp:
    def __init__(self):
        self.engine = QuakeEngine()
        self.engine.veri_cek_hazirla()
        self.engine.yapay_zeka_analizi()
        print("Sistem Hazır!")
    
    def calistir(self):
        while True:
            secim = input("[1] EDA Özetini Gör\n[2] Dünyadaki Anormal Depremleri Listele\n [3] Kendi Koordinatıma En Yakın Depremi Bul (Zorlu Görev!)\n[q] Çıkış")
            if secim == '1':
                print(self.engine.eda_ozeti())
            elif secim == '2':
                print(self.engine.anomallikleri_getir())
            elif secim == '3':
                enlem = float(input("Enlem giriniz"))
                boylam = float(input("Boylam giriniz"))
                fark = abs(self.engine.df['boylam'] - boylam) + abs(self.engine.df['enlem'] - enlem)
                self.engine.df['Fark'] = fark
                print(self.engine.df.sort_values(by='Fark', ascending=True).head(1))
            elif secim == 'q':
                break

In [48]:
# Sadece uygulamayı başlat! Gerisini 'while' döngüsü ve sen halledeceksin.
uygulama = TerminalApp()
uygulama.calistir()

Sistem Hazır!
     mag  sig   boylam    enlem  derinlik  Anomali_Durumu    Fark
391  4.7  340  38.7885  39.7189      10.0               1  8.1996
     mag  sig    boylam    enlem  derinlik  Anomali_Durumu      Fark
10   6.2  592   96.7387   2.0420    18.000              -1  103.8267
43   6.3  611 -179.4431 -21.9023   596.319              -1  273.6754
71   5.3  773   89.1394  22.4513     9.751              -1   75.8181
131  5.0  385 -177.6036 -20.1016   393.607              -1  270.0352
132  6.1  577 -169.8574  52.3572    14.000              -1  213.0846
142  7.1  791  116.2637   6.8285   619.849              -1  118.5652
151  5.3  432  179.4956 -21.8475   634.606              -1  210.4731
152  6.0  554  179.5487 -21.7999   653.785              -1  210.4786
158  4.5  312 -177.9408 -17.8995   605.972              -1  268.1703
171  5.4  449  172.4032 -13.8873   608.731              -1  195.4205
176  4.5  312  179.2416 -23.4845   558.638              -1  211.8561
194  4.5  312 -178.4225 -2